In [1]:
import os

from pathlib import Path
os.chdir(Path.cwd().parent)
print(f"Current directory: {Path.cwd()}")

Current directory: /opt/asaldivar/projects/h_lacustris


In [2]:
import cobra
import polars as pl

from memote.suite.tests.test_biomass import test_biomass_consistency

In [3]:
def fix_biomass_consistency(model, biomass_rxn_id):
    print(f"Testing reaction: {biomass_rxn_id}")
    biomass_rxn = model.reactions.get_by_id(biomass_rxn_id)
    mets_in_biomass = biomass_rxn.metabolites
    try:
        test_biomass_consistency(model, biomass_rxn_id)
    except AssertionError as e:
        print(e)
        mets_info = [(met.id, met.formula_weight, coeff) for met, coeff in mets_in_biomass.items()]
        mets_info_df = pl.DataFrame(
            mets_info,
            orient="row",
            schema=["met_id", "formula_weight", "coeff"]
        )
        new_coeffs = normalize_biomass_rxn(mets_info_df)
        biomass_rxn.subtract_metabolites(mets_in_biomass)
        biomass_rxn.add_metabolites(new_coeffs)
    else:
        print("Reaction is ok")

def normalize_biomass_rxn(mets_info_df):
    molecular_mass = 0
    for row in mets_info_df.iter_rows(named=True):
        molecular_mass += row["formula_weight"] * -row["coeff"]

    new_coeffs = {}
    new_mass = 0
    for row in mets_info_df.iter_rows(named=True):
        coeff = 1000 * row["coeff"] / molecular_mass
        new_mass += row["formula_weight"] * -coeff
        new_coeffs[row["met_id"]] = coeff

    print(f"The current molar mass is {molecular_mass / 1000}  mmol / g[CDW] / h")
    print(f"The new molar mass is {new_mass / 1000}  mmol / g[CDW] / h")

    return new_coeffs

In [4]:
model_dir = Path("models/draft/v0.0.1")
model_file = model_dir / "hlacustris.xml"
model = cobra.io.read_sbml_model(model_file)
model

Set parameter Username
Set parameter LicenseID to value 2744905
Academic license - for non-commercial use only - expires 2026-11-26


Name,iHlct
Memory address,7fbd62b7af90
Number of metabolites,1766
Number of reactions,2294
Number of genes,762
Number of groups,0
Objective expression,1.0*BIOMASS_hlacus_auto - 1.0*BIOMASS_hlacus_auto_reverse_375bd
Compartments,"mitochondrion, chloroplast, cytoplasm, glyoxysome, extracellular, thylakoid"


In [5]:
sol = model.optimize()
model.summary(solution=sol)

Metabolite,Reaction,Flux,C-Number,C-Flux
co2_e,EX_co2_e,156.1,1,100.00%
h2o_e,EX_h2o_e,126.5,0,0.00%
h_e,EX_h_e,0.5801,0,0.00%
mg2_e,EX_mg2_e,0.01338,0,0.00%
no3_e,EX_no3_e,1.362,0,0.00%
o2_e,EX_o2_e,1.188,0,0.00%
photonVis_e,EX_photonVis_e,1000,0,0.00%
pi_e,EX_pi_e,0.5083,0,0.00%
so4_e,EX_so4_e,0.0718,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux


In [6]:
print("These mets don't have formula weight\n")
for met in model.reactions.BIOMASS_hlacus_auto.metabolites:
    if not met.formula:
        print(met)

These mets don't have formula weight



In [7]:
biomass_rxn_id = "BIOMASS_hlacus_auto"
fix_biomass_consistency(model, biomass_rxn_id)

Testing reaction: BIOMASS_hlacus_auto
The component molar mass of the biomass reaction BIOMASS_hlacus_auto
sums up to 8.05015986123145             which is outside of the 1e-03
margin from 1 mmol / g[CDW] / h.
The current molar mass is 8.050159861231442  mmol / g[CDW] / h
The new molar mass is 1.0000000000000053  mmol / g[CDW] / h


In [8]:
sol = model.optimize()
model.summary(solution=sol)

Metabolite,Reaction,Flux,C-Number,C-Flux
co2_e,EX_co2_e,156.1,1,100.00%
h2o_e,EX_h2o_e,126.5,0,0.00%
h_e,EX_h_e,0.5801,0,0.00%
mg2_e,EX_mg2_e,0.01338,0,0.00%
no3_e,EX_no3_e,1.362,0,0.00%
o2_e,EX_o2_e,1.188,0,0.00%
photonVis_e,EX_photonVis_e,1000,0,0.00%
pi_e,EX_pi_e,0.5083,0,0.00%
so4_e,EX_so4_e,0.0718,0,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux


In [9]:
cobra.io.write_sbml_model(model, model_file)